# 03 — GraphConv (GCN) Classifier

학습 → 평가 → per-task ROC curve 12개. 결과는 `results/figures`와 Drive에 저장.

In [ ]:
# === Colab setup (run once, restart runtime if prompted) ===
import os, sys
if not os.path.exists("tox21-toxicity-classifier"):
    !git clone https://github.com/zqvo04/tox21-toxicity-classifier.git
%cd tox21-toxicity-classifier
!bash setup_colab.sh
# Make src importable
sys.path.insert(0, os.getcwd())

In [ ]:
# Mount Google Drive for checkpoints (edit path to your own folder)
from google.colab import drive
drive.mount('/content/drive')
CKPT_DIR = '/content/drive/MyDrive/tox21/'
os.makedirs(CKPT_DIR, exist_ok=True)

In [ ]:
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from src import losses, models, train, evaluate
FIG = 'results/figures'; os.makedirs(FIG, exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'; print('device:', device)

## 1. 데이터 로딩

In [ ]:
from src.dataset import make_graph_dataloaders
train_loader, valid_loader, test_loader, tasks, node_dim = make_graph_dataloaders(batch_size=64)
print('tasks:', tasks, '| node_dim:', node_dim)

## 2. 손실함수 (pos_weight 자동 계산) & 모델

In [ ]:
# pos_weight from training labels (NaN-aware)
import numpy as np
from src.dataset import load_tox21_graph
_, (tr_ds, _, _), _ = load_tox21_graph()
pos_weight = losses.compute_pos_weight(tr_ds.y, tr_ds.w)
print('pos_weight:', np.round(pos_weight.numpy(), 2))
loss_fn = losses.build_loss('bce', pos_weight=pos_weight)

In [ ]:
model = models.build_model('gcn', node_dim=node_dim, hidden=128, n_tasks=len(tasks), dropout=0.3)
print(model)

## 3. 학습 (Early stopping patience=10)

In [ ]:
cfg = train.TrainConfig(ckpt_dir=CKPT_DIR, ckpt_name='graphconv_best.pt',
                        max_epochs=100, patience=10, lr=1e-3)
trainer = train.Trainer(model, loss_fn, train.graph_forward,
                        config=cfg, device=device)
history = trainer.fit(train_loader, valid_loader)
trainer.save_curves(f'{FIG}/graphconv_learning_curve.png',
                    title='graphconv learning curve')

## 4. 평가 (test set): per-task ROC-AUC / PR-AUC / F1

In [ ]:
probs, y_true, mask = trainer.predict(test_loader)
df_metrics, summary = evaluate.evaluate(y_true, probs, mask, tasks, name='graphconv')
print('Summary:', summary)
df_metrics.to_csv(f'{FIG}/graphconv_metrics.csv', index=False)
display(df_metrics.round(4))

## 5. ROC curve 12개

In [ ]:
evaluate.plot_roc_curves(y_true, probs, mask, tasks,
                         save_path=f'{FIG}/graphconv_roc_curves.png',
                         title='graphconv: per-task ROC')
plt.show()
print('Mean ROC-AUC: %.4f' % summary['mean_roc_auc'])

In [ ]:
# Persist predictions for the comparison notebook
np.savez(f'{CKPT_DIR}/graphconv_preds.npz',
         probs=probs, y_true=y_true, mask=mask, tasks=np.array(tasks, dtype=object))
print('saved predictions to Drive')